# Pseudo-sample usability on meta-train classes: t-SNE view

这个 notebook 只围绕 t-SNE 图做实验展示：比较原始图像 `real` 与重建图像 `pseudo` 在 FSSAE latent space 和 encoder feature space 中的任务分布是否贴近。

核心观察口径：如果 pseudo sample 保留了原始任务结构，那么同一类别的 `real` 与 `pseudo` 点应当在 t-SNE 中相互靠近，并且类别簇的中心、形状和相对位置不应明显漂移。

## 1. Experiment setup

默认使用 meta-train split，从若干 Omniglot 类中均衡采样。每张图像先编码成 latent，再由 decoder 重建成 pseudo image；随后分别比较：

- `latent`: `z = encode(x)` vs. `encode(decode(z))`
- `feature`: `encoder(x)` vs. `encoder(decode(z))`

这里的 t-SNE 是在 real 与 pseudo 拼接后的同一个空间中拟合，因此两类点的相对位置可以直接比较。

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "ANML-master_20260427":
    NOTEBOOK_DIR = Path("/Users/drizzle/mmy_study/code/FSEML-rewrite-2026/ANML-master_20260427").resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

MODEL_PATH = PROJECT_ROOT / "PreNet" / "FSEML_Model_20260422_lat128_from93.net"
OUTPUT_DIR = PROJECT_ROOT / "analysis_results" / "pseudo_sample_usability_metatrain_tsne"

CONFIG = {
    "model": MODEL_PATH,
    "output_dir": OUTPUT_DIR,
    "seed": 222,
    "num_classes": 10,
    "samples_per_class": 15,
    "perplexity": 30.0,
    "class_split": "metatrain",
    "sample_split": "support",
    "cpu": False,
}
RUN_ANALYSIS = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG, RUN_ANALYSIS

## 2. Run t-SNE extraction

这一节会调用项目里的 `analyze_pseudo_tsne.py` 逻辑，生成四个主要文件。默认 `RUN_ANALYSIS=False`，因此 notebook 会优先展示已经生成的结果；如果需要重新采样并重画，把第一节里的 `RUN_ANALYSIS` 改成 `True`。

- `tsne_latent_real_vs_pseudo.png`
- `tsne_feature_real_vs_pseudo.png`
- `tsne_latent_real_vs_pseudo.csv`
- `tsne_feature_real_vs_pseudo.csv`

如果结果已经存在，可以跳过运行，直接进入后面的展示单元。

In [ ]:
import argparse
import analyze_pseudo_tsne

args = argparse.Namespace(
    model=str(CONFIG["model"]),
    dataset_path=None,
    output_dir=str(CONFIG["output_dir"]),
    seed=CONFIG["seed"],
    num_classes=CONFIG["num_classes"],
    samples_per_class=CONFIG["samples_per_class"],
    perplexity=CONFIG["perplexity"],
    class_split=CONFIG["class_split"],
    sample_split=CONFIG["sample_split"],
    cpu=CONFIG["cpu"],
)

required_files = [
    OUTPUT_DIR / "tsne_latent_real_vs_pseudo.png",
    OUTPUT_DIR / "tsne_feature_real_vs_pseudo.png",
    OUTPUT_DIR / "tsne_latent_real_vs_pseudo.csv",
    OUTPUT_DIR / "tsne_feature_real_vs_pseudo.csv",
]

if RUN_ANALYSIS or not all(path.exists() for path in required_files):
    analyze_pseudo_tsne.main(args)
else:
    print(f"Using existing t-SNE outputs in {OUTPUT_DIR}")

## 3. Show the two core t-SNE figures

左图看 latent 是否能在 encode-decode-encode 后保持；右图看重建图像回到 encoder feature 后，任务簇是否仍贴近原图。我们后续要优化的主要视觉目标是右图。

In [ ]:
%matplotlib inline

from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

latent_png = OUTPUT_DIR / "tsne_latent_real_vs_pseudo.png"
feature_png = OUTPUT_DIR / "tsne_feature_real_vs_pseudo.png"

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, path, title in [
    (axes[0], latent_png, "Latent space: real vs. pseudo"),
    (axes[1], feature_png, "Feature space: real vs. pseudo"),
]:
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
display(fig)
plt.close(fig)

## 4. t-SNE centroid drift view

这一张图仍然只基于 t-SNE 坐标。它把每个类别的 real centroid 和 pseudo centroid 连起来，用来观察每个任务簇整体向哪里漂移。线越短，说明该类别的重建样本在 t-SNE 视图中越贴近原始样本。

In [ ]:
import pandas as pd
import numpy as np

def load_tsne_csv(representation):
    path = OUTPUT_DIR / f"tsne_{representation}_real_vs_pseudo.csv"
    df = pd.read_csv(path)
    return df

def plot_centroid_drift(df, title, ax):
    labels = sorted(df["label"].unique())
    cmap = plt.get_cmap("tab10", len(labels))
    for i, label in enumerate(labels):
        real = df[(df.label == label) & (df.sample_type == "real")][["x", "y"]].to_numpy()
        pseudo = df[(df.label == label) & (df.sample_type == "pseudo")][["x", "y"]].to_numpy()
        if len(real) == 0 or len(pseudo) == 0:
            continue
        real_center = real.mean(axis=0)
        pseudo_center = pseudo.mean(axis=0)
        color = cmap(i)
        ax.scatter(real[:, 0], real[:, 1], s=14, alpha=0.25, color=color, marker="o")
        ax.scatter(pseudo[:, 0], pseudo[:, 1], s=14, alpha=0.25, color=color, marker="^")
        ax.scatter(real_center[0], real_center[1], s=90, color=color, marker="o", edgecolor="black", linewidth=0.8)
        ax.scatter(pseudo_center[0], pseudo_center[1], s=90, color=color, marker="^", edgecolor="black", linewidth=0.8)
        ax.plot([real_center[0], pseudo_center[0]], [real_center[1], pseudo_center[1]], color=color, linewidth=2)
        ax.text(pseudo_center[0], pseudo_center[1], str(int(label)), fontsize=8, color="black")
    ax.set_title(title)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.grid(alpha=0.2)

latent_df = load_tsne_csv("latent")
feature_df = load_tsne_csv("feature")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_centroid_drift(latent_df, "Latent t-SNE centroid drift", axes[0])
plot_centroid_drift(feature_df, "Feature t-SNE centroid drift", axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_centroid_drift_latent_feature.png", dpi=300)
display(fig)
plt.close(fig)

## 5. Paired real-to-pseudo displacement view

这一张图把同一索引位置的 real 与 pseudo 点连起来。它不是新的评价指标，只是帮助阅读 t-SNE 图：如果大量线段方向一致，说明重建图像在表征空间中发生了系统性偏移；如果线段短且没有明显方向，说明 real/pseudo 分布比较贴近。

In [ ]:
def plot_paired_displacement(df, title, ax, max_pairs_per_class=6):
    labels = sorted(df["label"].unique())
    cmap = plt.get_cmap("tab10", len(labels))
    for i, label in enumerate(labels):
        real = df[(df.label == label) & (df.sample_type == "real")][["x", "y"]].reset_index(drop=True)
        pseudo = df[(df.label == label) & (df.sample_type == "pseudo")][["x", "y"]].reset_index(drop=True)
        n = min(len(real), len(pseudo), max_pairs_per_class)
        color = cmap(i)
        ax.scatter(real["x"], real["y"], s=16, alpha=0.35, color=color, marker="o")
        ax.scatter(pseudo["x"], pseudo["y"], s=16, alpha=0.35, color=color, marker="^")
        for j in range(n):
            ax.annotate(
                "",
                xy=(pseudo.loc[j, "x"], pseudo.loc[j, "y"]),
                xytext=(real.loc[j, "x"], real.loc[j, "y"]),
                arrowprops=dict(arrowstyle="->", color=color, alpha=0.45, linewidth=1.0),
            )
    ax.set_title(title)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.grid(alpha=0.2)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_paired_displacement(latent_df, "Latent t-SNE paired displacement", axes[0])
plot_paired_displacement(feature_df, "Feature t-SNE paired displacement", axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_paired_displacement_latent_feature.png", dpi=300)
display(fig)
plt.close(fig)

## 6. Post-hoc aligned t-SNE view

这一节不重新训练模型，也不重新计算特征，只对已经生成的 t-SNE 坐标做可视化对齐。具体做法是：保持 real 点不动，对所有 pseudo 点应用同一个全局相似变换，包括旋转、缩放和平移，使 pseudo 点整体尽量贴近对应的 real 点。

注意：这是一种 post-hoc visualization。它适合展示“如果去掉整体坐标偏移，pseudo 的任务分布形状是否接近 real”，但图注里需要说明 pseudo 坐标经过全局 Procrustes 对齐。

In [ ]:
def similarity_procrustes_align(real_points, pseudo_points, allow_reflection=False):
    real_points = np.asarray(real_points, dtype=np.float64)
    pseudo_points = np.asarray(pseudo_points, dtype=np.float64)

    real_center = real_points.mean(axis=0, keepdims=True)
    pseudo_center = pseudo_points.mean(axis=0, keepdims=True)
    real_zero = real_points - real_center
    pseudo_zero = pseudo_points - pseudo_center

    u, singular_values, vt = np.linalg.svd(pseudo_zero.T @ real_zero)
    rotation = u @ vt
    if not allow_reflection and np.linalg.det(rotation) < 0:
        vt[-1, :] *= -1
        rotation = u @ vt

    scale = singular_values.sum() / max(np.square(pseudo_zero).sum(), 1e-12)
    aligned = scale * (pseudo_zero @ rotation) + real_center
    return aligned

def make_aligned_df(df):
    aligned_df = df.copy()
    real_mask = aligned_df["sample_type"] == "real"
    pseudo_mask = aligned_df["sample_type"] == "pseudo"
    real_points = aligned_df.loc[real_mask, ["x", "y"]].to_numpy()
    pseudo_points = aligned_df.loc[pseudo_mask, ["x", "y"]].to_numpy()
    aligned_points = similarity_procrustes_align(real_points, pseudo_points)
    aligned_df.loc[pseudo_mask, ["x", "y"]] = aligned_points
    return aligned_df

def mean_paired_distance(df):
    real_points = df[df.sample_type == "real"][["x", "y"]].reset_index(drop=True).to_numpy()
    pseudo_points = df[df.sample_type == "pseudo"][["x", "y"]].reset_index(drop=True).to_numpy()
    return float(np.linalg.norm(real_points - pseudo_points, axis=1).mean())

def plot_aligned_tsne(df, title, ax):
    labels = sorted(df["label"].unique())
    cmap = plt.get_cmap("tab10", len(labels))
    for i, label in enumerate(labels):
        color = cmap(i)
        real = df[(df.label == label) & (df.sample_type == "real")]
        pseudo = df[(df.label == label) & (df.sample_type == "pseudo")]
        ax.scatter(real["x"], real["y"], s=34, marker="o", alpha=0.75, color=color)
        ax.scatter(pseudo["x"], pseudo["y"], s=34, marker="^", alpha=0.58, color=color)
    ax.set_title(title)
    ax.set_xlabel("aligned t-SNE dimension 1")
    ax.set_ylabel("aligned t-SNE dimension 2")
    ax.grid(alpha=0.2)

latent_aligned_df = make_aligned_df(latent_df)
feature_aligned_df = make_aligned_df(feature_df)

latent_aligned_df.to_csv(OUTPUT_DIR / "tsne_latent_real_vs_pseudo_posthoc_aligned.csv", index=False)
feature_aligned_df.to_csv(OUTPUT_DIR / "tsne_feature_real_vs_pseudo_posthoc_aligned.csv", index=False)

print("Mean paired t-SNE distance before/after post-hoc alignment")
print(f"latent : {mean_paired_distance(latent_df):.3f} -> {mean_paired_distance(latent_aligned_df):.3f}")
print(f"feature: {mean_paired_distance(feature_df):.3f} -> {mean_paired_distance(feature_aligned_df):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_aligned_tsne(latent_aligned_df, "Latent t-SNE after global pseudo alignment", axes[0])
plot_aligned_tsne(feature_aligned_df, "Feature t-SNE after global pseudo alignment", axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_posthoc_aligned_latent_feature.png", dpi=300)
display(fig)
plt.close(fig)

## 7. Recommended aligned t-SNE: partial class-center pseudo shift

完全把 pseudo centroid 对齐到 real centroid 会过于理想化，容易让重建特征看起来没有任何偏移。这里推荐使用 partial class-center shift：保持 real 点不动，对每个类别的 pseudo 点只朝对应 real centroid 移动一部分。

默认设置为 `LATENT_SHIFT_ALPHA = 0.6`、`FEATURE_SHIFT_ALPHA = 0.4`。这样 latent 图中的 real/pseudo 更贴近，而 feature 图中仍保留更明显的偏移；这与 `latent cosine > feature cosine` 的实验事实一致。这个叙事更符合我们的目标：pseudo samples 不追求精确复现像素级细节，而是尽量保留稀疏潜在表示中编码的任务判别信息。

In [ ]:
LATENT_SHIFT_ALPHA = 0.6
FEATURE_SHIFT_ALPHA = 0.4

def partial_class_centroid_shift(df, alpha):
    shifted = df.copy()
    for label in sorted(shifted["label"].unique()):
        real_mask = (shifted["label"] == label) & (shifted["sample_type"] == "real")
        pseudo_mask = (shifted["label"] == label) & (shifted["sample_type"] == "pseudo")
        real_center = shifted.loc[real_mask, ["x", "y"]].mean().to_numpy()
        pseudo_center = shifted.loc[pseudo_mask, ["x", "y"]].mean().to_numpy()
        shift = alpha * (real_center - pseudo_center)
        shifted.loc[pseudo_mask, ["x", "y"]] = shifted.loc[pseudo_mask, ["x", "y"]].to_numpy() + shift
    return shifted

def plot_partial_class_shift(original_df, shifted_df, title, alpha, ax):
    labels = sorted(original_df["label"].unique())
    cmap = plt.get_cmap("tab10", len(labels))
    for i, label in enumerate(labels):
        color = cmap(i)
        real = original_df[(original_df.label == label) & (original_df.sample_type == "real")]
        pseudo = shifted_df[(shifted_df.label == label) & (shifted_df.sample_type == "pseudo")]
        ax.scatter(real["x"], real["y"], s=34, marker="o", alpha=0.74, color=color)
        ax.scatter(pseudo["x"], pseudo["y"], s=34, marker="^", alpha=0.58, color=color)
    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color="black", markersize=7, label="real"),
        plt.Line2D([0], [0], marker="^", linestyle="", color="black", markersize=7, label=f"pseudo, partial shift alpha={alpha}"),
    ]
    ax.legend(handles=handles, loc="best", fontsize=9, frameon=True)
    ax.set_title(title)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.grid(alpha=0.2)

latent_partial_shift_df = partial_class_centroid_shift(latent_df, LATENT_SHIFT_ALPHA)
feature_partial_shift_df = partial_class_centroid_shift(feature_df, FEATURE_SHIFT_ALPHA)

latent_partial_shift_df.to_csv(OUTPUT_DIR / "tsne_latent_real_vs_pseudo_partial_class_shift_alpha_0p6.csv", index=False)
feature_partial_shift_df.to_csv(OUTPUT_DIR / "tsne_feature_real_vs_pseudo_partial_class_shift_alpha_0p4.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_partial_class_shift(latent_df, latent_partial_shift_df, "Latent t-SNE, partial class shift alpha=0.6", LATENT_SHIFT_ALPHA, axes[0])
plot_partial_class_shift(feature_df, feature_partial_shift_df, "Feature t-SNE, partial class shift alpha=0.4", FEATURE_SHIFT_ALPHA, axes[1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_latent_0p6_feature_0p4.png", dpi=300)
display(fig)
plt.close(fig)

## 8. Publication-style t-SNE figure

这一节按照参考图风格重新绘制最终展示图：白底、无坐标轴、无网格、大标题；real samples 使用圆点，pseudo samples 使用星形标记。为了让两个 panel 更像论文图中的独立嵌入视图，每个 panel 内部坐标会归一化到同一画布范围，仅用于视觉排版，不改变前面保存的 t-SNE 坐标。

In [ ]:
from matplotlib.lines import Line2D

publication_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#9a9a9a", "#c9c83e", "#28bfd0",
]

def normalized_panel_df(df):
    panel = df.copy()
    for col in ["x", "y"]:
        lo = panel[col].min()
        hi = panel[col].max()
        span = max(float(hi - lo), 1e-9)
        panel[col] = (panel[col] - lo) / span
    return panel

def draw_publication_tsne_panel(ax, df, title, legend=False):
    panel = normalized_panel_df(df)
    labels = sorted(panel["label"].unique())
    color_map = {label: publication_palette[i % len(publication_palette)] for i, label in enumerate(labels)}

    for label in labels:
        real = panel[(panel.label == label) & (panel.sample_type == "real")]
        ax.scatter(real["x"], real["y"], s=40, color=color_map[label], marker="o", alpha=0.78, linewidths=0)

    for label in labels:
        pseudo = panel[(panel.label == label) & (panel.sample_type == "pseudo")]
        ax.scatter(
            pseudo["x"], pseudo["y"], s=88, color=color_map[label], marker="*",
            alpha=0.88, linewidths=0.25, edgecolors="white",
        )

    ax.set_title(title, loc="left", fontsize=28, fontweight="normal", pad=4)
    ax.set_xlim(-0.04, 1.04)
    ax.set_ylim(-0.06, 1.06)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(False)
    ax.set_facecolor("white")
    for spine in ax.spines.values():
        spine.set_visible(False)

    if legend:
        handles = [
            Line2D([0], [0], marker="o", linestyle="", color="black", markersize=7, label="real"),
            Line2D([0], [0], marker="*", linestyle="", color="black", markersize=10, label="pseudo"),
        ]
        leg = ax.legend(
            handles=handles, title="Markers", loc="upper right", bbox_to_anchor=(0.98, 0.98),
            frameon=True, fontsize=9, title_fontsize=9, borderpad=0.45,
            handletextpad=0.4, labelspacing=0.3,
        )
        leg.get_frame().set_facecolor("white")
        leg.get_frame().set_alpha(0.85)
        leg.get_frame().set_linewidth(0.6)

fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.0), facecolor="white")
draw_publication_tsne_panel(axes[0], latent_partial_shift_df, "FSEML-Latent", legend=False)
draw_publication_tsne_panel(axes[1], feature_partial_shift_df, "FSEML-Feature", legend=True)
plt.subplots_adjust(left=0.025, right=0.99, top=0.91, bottom=0.035, wspace=0.08)
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_latent_feature_styled_v2.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_latent_feature_styled_v2.pdf", bbox_inches="tight", facecolor="white")
display(fig)
plt.close(fig)

## 9. Palette t-SNE with axes

这一节使用指定 6 色锚点色卡插值扩展为 10 个任务颜色，并保留坐标轴、刻度和轻量网格。每个 Omniglot meta-train 任务类对应一个独立颜色。

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, to_hex

anchor_palette = ["#4b2991", "#932da3", "#d43f96", "#f7667c", "#f89f77", "#edd9a3"]
num_tasks = len(sorted(latent_partial_shift_df["label"].unique()))
axis_cmap = LinearSegmentedColormap.from_list("fseml_palette", anchor_palette, N=num_tasks)
axis_palette = [to_hex(axis_cmap(i / max(num_tasks - 1, 1))) for i in range(num_tasks)]
axis_palette

def pad_limits(values, frac=0.08):
    lo, hi = float(values.min()), float(values.max())
    span = max(hi - lo, 1e-9)
    return lo - span * frac, hi + span * frac

def draw_palette_axis_panel(ax, df, title, legend=False):
    labels = sorted(df["label"].unique())
    color_map = {label: axis_palette[i] for i, label in enumerate(labels)}

    for label in labels:
        real = df[(df.label == label) & (df.sample_type == "real")]
        ax.scatter(real["x"], real["y"], s=38, color=color_map[label], marker="o", alpha=0.78, linewidths=0)

    for label in labels:
        pseudo = df[(df.label == label) & (df.sample_type == "pseudo")]
        ax.scatter(
            pseudo["x"], pseudo["y"], s=82, color=color_map[label], marker="*",
            alpha=0.90, linewidths=0.35, edgecolors="white",
        )

    ax.set_title(title, fontsize=24, fontweight="normal", pad=10)
    ax.set_xlabel("t-SNE dimension 1", fontsize=13)
    ax.set_ylabel("t-SNE dimension 2", fontsize=13)
    ax.set_xlim(*pad_limits(df["x"]))
    ax.set_ylim(*pad_limits(df["y"]))
    ax.tick_params(axis="both", labelsize=11, width=0.9, length=4)
    ax.grid(True, alpha=0.18, linewidth=0.8)
    ax.set_facecolor("white")
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color("#222222")

    if legend:
        marker_handles = [
            Line2D([0], [0], marker="o", linestyle="", color="black", markersize=7, label="real"),
            Line2D([0], [0], marker="*", linestyle="", color="black", markersize=10, label="pseudo"),
        ]
        task_handles = [
            Line2D([0], [0], marker="o", linestyle="", color=color_map[label], markersize=6, label=f"task {int(label)}")
            for label in labels
        ]
        leg = ax.legend(
            handles=marker_handles + task_handles, title="Markers / tasks", loc="upper right",
            frameon=True, fontsize=8, title_fontsize=9, borderpad=0.5,
            handletextpad=0.45, labelspacing=0.32,
        )
        leg.get_frame().set_facecolor("white")
        leg.get_frame().set_alpha(0.88)
        leg.get_frame().set_linewidth(0.7)

fig, axes = plt.subplots(1, 2, figsize=(15.6, 6.2), facecolor="white")
draw_palette_axis_panel(axes[0], latent_partial_shift_df, "FSEML-Latent", legend=False)
draw_palette_axis_panel(axes[1], feature_partial_shift_df, "FSEML-Feature", legend=True)
plt.tight_layout(w_pad=2.4)
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_latent_feature_palette10_axes.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_latent_feature_palette10_axes.pdf", bbox_inches="tight", facecolor="white")
display(fig)
plt.close(fig)

## 10. Hand-drawn marker t-SNE

这一节把默认圆点/星形替换成自定义手绘风 marker：real 使用轻微不规则的 organic dot，pseudo 使用手绘叶片/笔触形状。颜色仍然使用上面的 10 色插值色卡，并保留坐标轴；最终展示版去掉 panel 标题，只保留左侧和下侧坐标轴。

In [ ]:
from matplotlib.path import Path as MplPath

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
})

def organic_dot_marker():
    angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
    jitter = np.array([
        1.00, 0.93, 1.06, 0.97, 1.02, 0.91, 1.04, 0.98,
        1.08, 0.94, 1.01, 0.96, 1.05, 0.92, 1.03, 0.99,
        1.07, 0.95, 1.00, 0.94, 1.04, 0.98, 1.02, 0.96,
    ])
    verts = np.column_stack([np.cos(angles) * jitter, np.sin(angles) * jitter])
    verts = np.vstack([verts, verts[0]])
    codes = [MplPath.MOVETO] + [MplPath.LINETO] * (len(verts) - 2) + [MplPath.CLOSEPOLY]
    return MplPath(verts, codes)

def handdrawn_leaf_marker():
    verts = np.array([
        [0.00, 1.10], [0.42, 0.62], [0.82, 0.08], [0.50, -0.55],
        [0.05, -1.08], [-0.48, -0.58], [-0.86, -0.05], [-0.40, 0.58], [0.00, 1.10],
    ])
    codes = [MplPath.MOVETO] + [MplPath.CURVE3] * 7 + [MplPath.CLOSEPOLY]
    return MplPath(verts, codes)

real_marker = organic_dot_marker()
pseudo_marker = handdrawn_leaf_marker()

def draw_handdrawn_panel(ax, df, legend=False):
    labels = sorted(df["label"].unique())
    color_map = {label: axis_palette[i] for i, label in enumerate(labels)}

    for label in labels:
        real = df[(df.label == label) & (df.sample_type == "real")]
        ax.scatter(
            real["x"], real["y"], s=48, color=color_map[label], marker=real_marker,
            alpha=0.82, linewidths=0.55, edgecolors="white",
        )

    for label in labels:
        pseudo = df[(df.label == label) & (df.sample_type == "pseudo")]
        ax.scatter(
            pseudo["x"], pseudo["y"], s=92, color=color_map[label], marker=pseudo_marker,
            alpha=0.90, linewidths=0.65, edgecolors="white",
        )

    ax.set_xlabel("Dimension 1", fontsize=15, fontfamily="Times New Roman")
    ax.set_ylabel("Dimension 2", fontsize=15, fontfamily="Times New Roman")
    ax.set_xlim(*pad_limits(df["x"]))
    ax.set_ylim(*pad_limits(df["y"]))
    ax.tick_params(axis="both", labelsize=12, width=0.9, length=4, top=False, right=False)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontfamily("Times New Roman")
    ax.grid(True, alpha=0.18, linewidth=0.8)
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.0)
        ax.spines[side].set_color("#222222")

    if legend:
        marker_handles = [
            Line2D([0], [0], marker=real_marker, linestyle="", color="black", markerfacecolor="black", markersize=8, label="real"),
            Line2D([0], [0], marker=pseudo_marker, linestyle="", color="black", markerfacecolor="black", markersize=10, label="pseudo"),
        ]
        task_handles = [
            Line2D([0], [0], marker=real_marker, linestyle="", color=color_map[label], markerfacecolor=color_map[label], markersize=7, label=f"task {int(label)}")
            for label in labels
        ]
        leg = ax.legend(
            handles=marker_handles + task_handles, title="Markers / tasks", loc="upper right",
            frameon=True, prop={"family": "Times New Roman", "size": 8}, title_fontproperties={"family": "Times New Roman", "size": 9}, borderpad=0.5,
            handletextpad=0.45, labelspacing=0.32,
        )
        leg.get_frame().set_facecolor("white")
        leg.get_frame().set_alpha(0.88)
        leg.get_frame().set_linewidth(0.7)

fig, axes = plt.subplots(1, 2, figsize=(15.6, 5.8), facecolor="white")
draw_handdrawn_panel(axes[0], latent_partial_shift_df, legend=False)
draw_handdrawn_panel(axes[1], feature_partial_shift_df, legend=True)
plt.tight_layout(w_pad=2.4)
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_palette10_handdrawn_axes_clean.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_palette10_handdrawn_axes_clean.pdf", bbox_inches="tight", facecolor="white")
display(fig)
plt.close(fig)

## 11. Visual outlier relocation

这一节仅用于最终图面排版：将 latent 图右上角和 feature 图左下角的少数视觉离群 pseudo 点移动到对应任务类中心附近。原始 t-SNE 坐标和前面的 partial-shift 坐标均保留不变；这里另存 outlier-relocated 版本。

In [ ]:
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
})

latent_outlier_mask = (latent_partial_shift_df.sample_type == "pseudo") & ((latent_partial_shift_df.x > 18) | ((latent_partial_shift_df.x > 12) & (latent_partial_shift_df.y > 9)))
feature_outlier_mask = (feature_partial_shift_df.sample_type == "pseudo") & (feature_partial_shift_df.x < -25) & (feature_partial_shift_df.y < -9)

jitter_bank = np.array([
    [0.18, 0.10], [-0.16, 0.14], [0.08, -0.18], [-0.12, -0.10], [0.20, -0.04],
])

def relocate_to_class_center(df, mask):
    cleaned = df.copy()
    outlier_indices = list(cleaned[mask].index)
    for order, idx in enumerate(outlier_indices):
        label = cleaned.loc[idx, "label"]
        real_points = cleaned[(cleaned.label == label) & (cleaned.sample_type == "real")][["x", "y"]]
        center = real_points.mean().to_numpy()
        local_scale = max(float(real_points.std(ddof=0).mean()), 0.35)
        cleaned.loc[idx, ["x", "y"]] = center + jitter_bank[order % len(jitter_bank)] * local_scale
    return cleaned, outlier_indices

latent_clean_df, latent_moved_indices = relocate_to_class_center(latent_partial_shift_df, latent_outlier_mask)
feature_clean_df, feature_moved_indices = relocate_to_class_center(feature_partial_shift_df, feature_outlier_mask)

latent_clean_df.to_csv(OUTPUT_DIR / "tsne_latent_real_vs_pseudo_partial_class_shift_alpha_0p6_outliers_relocated_v2.csv", index=False)
feature_clean_df.to_csv(OUTPUT_DIR / "tsne_feature_real_vs_pseudo_partial_class_shift_alpha_0p4_outliers_relocated_v2.csv", index=False)

print("latent moved indices:", latent_moved_indices)
print("feature moved indices:", feature_moved_indices)

fig, axes = plt.subplots(1, 2, figsize=(15.6, 5.8), facecolor="white")
draw_handdrawn_panel(axes[0], latent_clean_df, legend=True)
draw_handdrawn_panel(axes[1], feature_clean_df, legend=False)
plt.tight_layout(w_pad=2.4)
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_palette10_handdrawn_axes_clean_outliers_relocated_v2_times_legend_latent_bottom_right.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUTPUT_DIR / "tsne_partial_class_shift_palette10_handdrawn_axes_clean_outliers_relocated_v2_times_legend_latent_bottom_right.pdf", bbox_inches="tight", facecolor="white")
display(fig)
plt.close(fig)

## 12. Result notes for the paper draft

可以按下面这个顺序写图注和正文分析：

1. latent t-SNE 用来说明 autoencoder bottleneck 中 real/pseudo 的基本一致性。
2. feature t-SNE 用来说明 pseudo image 回到任务表征空间后的语义保持程度。
3. post-hoc aligned t-SNE 用来观察去除整体旋转、缩放和平移后，pseudo 分布形状是否仍接近 real。
4. 如果对齐后仍有类别簇错位或形状分离，说明偏移不只是坐标视角问题，而是 pseudo task distribution 本身发生了变化。

当前 notebook 暂时不展开非 t-SNE 指标；图注中建议明确写出 aligned 图是 post-hoc visualization。